# 03-hf-multitask-baseline.ipynb
---
## Multi-Task Learning for Joint Conspiracy Detection and Span Extraction

This notebook implements the multitask learning baseline for SemEval-2026 Task 10 (PsyCoMark).

### Architecture:
- **Shared Encoder:** `microsoft/deberta-v3-base`.
- **Task Heads:** 
  1.  **Token Classification Head (S1):** Predicts BIO labels for 5 marker types on each token.
  2.  **Sequence Classification Head (S2):** Predicts the document-level label (`conspiracy`/`non`).
- **Loss:** A combined loss `(Loss_span + Loss_doc)` is used to train the model end-to-end.

### Workflow:
1.  **Reproducibility & Config:** Set up seeds, logging, and a configuration dataclass.
2.  **Data Loading:** Load the cleaned `train.jsonl` and `dev.jsonl` from the `data_pipeline.py` output.
3.  **Token Alignment:** The core preprocessing step. Convert character-level span annotations into token-level, multi-hot BIO labels.
4.  **Custom Model:** Define a custom `DebertaV3ForMultiTask` class in PyTorch.
5.  **Custom Trainer:** Subclass the `transformers.Trainer` to handle the combined loss.
6.  **Metrics:** Implement a `compute_metrics` function to evaluate both subtasks simultaneously.
7.  **Training & Evaluation:** Run the training loop and save the final model, metrics, and predictions.

## SECTION 1: IMPORTS & ENVIRONMENT CAPTURE


In [ ]:

import os
import sys
import json
import logging
import subprocess
from pathlib import Path
from dataclasses import dataclass, field, asdict
from datetime import datetime

import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoConfig,
    PreTrainedModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from transformers.modeling_outputs import ModelOutput
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

# Suppress verbose logging from libraries
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
for lib in ["transformers", "datasets"]:
    logging.getLogger(lib).setLevel(logging.WARNING)

def set_seed(seed: int):
    """Sets the seed for reproducibility."""
    import random
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_git_commit_hash() -> str:
    """Gets the current git commit hash for logging."""
    try:
        return subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode('ascii').strip()
    except Exception:
        return "git_not_found"

c:\Users\panagiotis\Desktop\GitHub\PsyChoMark_Semeval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## SECTION 2: CONFIGURATION

In [2]:
# ==============================================================================
# ==============================================================================
@dataclass
class ModelConfig:
    # --- Paths and Metadata ---
    run_timestamp: str = field(default_factory=lambda: datetime.now().strftime("%Y%m%d_%H%M%S"))
    output_root: str = "./outputs/hf_multitask"
    
    # --- Model & Tokenizer ---
    model_name: str = "microsoft/deberta-v3-base"
    max_length: int = 1024
    
    # --- Training Hyperparameters ---
    seed: int = 42
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    num_train_epochs: int = 5
    per_device_train_batch_size: int = 4 # Small due to multitask complexity
    per_device_eval_batch_size: int = 8
    gradient_accumulation_steps: int = 4 # Effective batch size = 4 * 4 = 16
    fp16: bool = torch.cuda.is_available()
    
    # --- Task-specific ---
    loss_lambda_span: float = 1.0
    loss_lambda_doc: float = 1.0
    span_decision_threshold: float = 0.5
    
    def __post_init__(self):
        self.output_dir = str(Path(self.output_root) / self.run_timestamp)

# Instantiate config
config = ModelConfig()
set_seed(config.seed)

# Create output directory
Path(config.output_dir).mkdir(parents=True, exist_ok=True)
logging.info(f"Output will be saved to: {config.output_dir}")

2025-10-05 11:48:38,278 [INFO] Output will be saved to: outputs\hf_multitask\20251005_114838


## SECTION 3: DATA LOADING & CONSTANTS

In [3]:
latest_ptr = Path("./data/derived/psycomark_latest.txt")
if not latest_ptr.exists():
    raise FileNotFoundError("Run data_pipeline.py first!")
data_dir = Path(latest_ptr.read_text().strip())

train_path = data_dir / "train.jsonl"
dev_path = data_dir / "dev.jsonl"

logging.info(f"Loading data from {data_dir}")
raw_datasets = DatasetDict({
    "train": Dataset.from_json(str(train_path)),
    "dev": Dataset.from_json(str(dev_path))
})

# --- Constants & Mappings ---
SPAN_LABELS = sorted(["Actor", "Action", "Effect", "Victim", "Evidence"])
DOC_LABELS_MAP = {"non": 0, "conspiracy": 1}

# Create BIO labels and mappings
bio_span_labels = [f"B-{label}" for label in SPAN_LABELS] + [f"I-{label}" for label in SPAN_LABELS]
id2biolabel = {i: label for i, label in enumerate(bio_span_labels)}
biolabel2id = {label: i for i, label in id2biolabel.items()}
NUM_SPAN_LABELS = len(bio_span_labels)

logging.info(f"Span BIO Labels (total {NUM_SPAN_LABELS}): {bio_span_labels}")


2025-10-05 11:48:53,482 [INFO] Loading data from C:\Users\panagiotis\Desktop\GitHub\PsyChoMark_Semeval\data\derived\psycomark_official_split_20250928_232947
Generating train split: 3360 examples [00:00, 231794.30 examples/s]
Generating train split: 100 examples [00:00, 21262.82 examples/s]
2025-10-05 11:48:53,575 [INFO] Span BIO Labels (total 10): ['B-Action', 'B-Actor', 'B-Effect', 'B-Evidence', 'B-Victim', 'I-Action', 'I-Actor', 'I-Effect', 'I-Evidence', 'I-Victim']


## SECTION 4 & 5: TOKENIZATION AND LABEL ALIGNMENT (CORE LOGIC)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

def tokenize_and_align_labels(examples):
    # Tokenize the text
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=config.max_length,
        padding=False, # We'll pad later in the collator
        return_offsets_mapping=True
    )
    
    # Initialize labels
    all_token_labels = []
    all_doc_labels = []

    for i, doc_id in enumerate(examples['doc_id']):
        offset_mapping = tokenized_inputs["offset_mapping"][i]
        sequence_len = len(offset_mapping)
        
        # --- S1: Align Span Labels ---
        token_labels = np.zeros((sequence_len, NUM_SPAN_LABELS), dtype=np.float32)
        markers = examples['markers'][i] or []
        
        for marker in markers:
            char_start, char_end, label = marker['start'], marker['end'], marker['label']
            if char_start is None or char_end is None or label not in SPAN_LABELS:
                continue

            b_label_id = biolabel2id[f"B-{label}"]
            i_label_id = biolabel2id[f"I-{label}"]

            is_first_token = True
            for token_idx, (start, end) in enumerate(offset_mapping):
                # If the token is special (CLS, SEP) or has zero length, skip
                if start == end:
                    continue
                
                # Check for overlap between token and marker
                if max(start, char_start) < min(end, char_end):
                    if is_first_token:
                        token_labels[token_idx, b_label_id] = 1.0
                        is_first_token = False
                    else:
                        token_labels[token_idx, i_label_id] = 1.0
        all_token_labels.append(token_labels)

        # --- S2: Align Doc Labels ---
        doc_label_str = examples['doc_label'][i]
        doc_label_int = DOC_LABELS_MAP.get(doc_label_str, -1) # -1 for 'cant_tell'
        all_doc_labels.append(doc_label_int)

    tokenized_inputs["token_labels"] = all_token_labels
    tokenized_inputs["doc_label"] = all_doc_labels
    return tokenized_inputs

# Filter out 'cant_tell' and apply the preprocessing
filtered_datasets = raw_datasets.filter(lambda x: x['doc_label'] in DOC_LABELS_MAP)

logging.info("Tokenizing and aligning labels...")
processed_datasets = filtered_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)
logging.info(f"Dataset structure:\n{processed_datasets}")

c:\Users\panagiotis\Desktop\GitHub\PsyChoMark_Semeval\.venv\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Filter: 100%|██████████| 100/100 [00:00<00:00, 30021.50 examples/s]
2025-10-05 11:49:17,674 [INFO] Tokenizing and aligning labels...
Map: 100%|██████████| 2759/2759 [00:00<00:00, 4663.50 examples/s]
2025-10-05 11:49:18,287 [INFO] Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['doc_label', 'input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'token_labels'],
        num_rows: 2759
    })
    dev: Dataset({
        feature